# 1. Obtención de datos de PokeAPI

In [2]:
#Importar librerias necesarias
import requests
import pandas as pd
import time

In [3]:
#Veamos el total de pokemons existentes en pokeAPI
test_response = requests.get("https://pokeapi.co/api/v2/pokemon-species").json()
total_pokemon_existentes = test_response["count"]
print(total_pokemon_existentes)

1025


In [ ]:
#Obtención de datos de PokeAPI
# 0. Configuración inicial
## Inicializamos el diccionario vacío donde acumularemos los datos de todas las variables de interés
datos_pokeapi = {
    "id": [],
    "name": [],
    "generation": [],
    "shape": [],
    "capture_rate":[],
    "height":[],
    "weight":[],
    "stat_hp":[],
    "stat_attack":[],
    "stat_special-attack":[],
    "stat_defense":[],
    "stat_special-defense":[],
    "stat_speed":[],
    "type_1":[],
    "type_2":[],
}

# Definimos headers globales y params(limits) para la primera llamada
headers = {"User-Agent": "MiProyectoEDA (contacto: tu_correo@ejemplo.com)"}
params = {"limit": 1025} #Total de pokemons

# Definimos urls de endpoints de pokeapi
url_pokemon_species = "https://pokeapi.co/api/v2/pokemon-species" #URL endpoint Pokemon Species
url_pokemon = "https://pokeapi.co/api/v2/pokemon" #URL endpoint Pokemon

#1. Solicitamos la lista de especies
# Creamos sesión global para que vaya más rápido y poder reutilizarla en las otras llamadas
session = requests.Session()
session.headers.update(headers) #Le damos los headers globales

print("1. Solicitando lista de especies a PokeAPI...")
species_results = session.get(url_pokemon_species, params=params, timeout=10).json()["results"]

#2. Obtenemos datos pero solo de los pokemon de forma base (criterio de selección)
print("2. Procesando y filtrando Pokémon de Nivel 1 (Formas Base)...")
for species in species_results:
    #2.1. Hacemos request de endpoint pokemon-species
    try:
        #Intentamos hacer el request de la url
        res_pokemon_sp = session.get(species["url"],timeout=3).json()
    except requests.exceptions.RequestException as e:
        print(f"Error de red del endpoint 'pokemon-species' en la especie {species['name']}, saltando... \nDetalle: {e}")
        continue #Si falla que salte al siguiente elemento del bucle
    else:
        #Comprobamos el nivel evolutivo (para nivel base, esta variable esta vacia)
        if res_pokemon_sp["evolves_from_species"] is None:
            #2.2. Hacemos el request del endpoint pokemon
            try:
                #Intentamos hacer el request de la url aprovechando id
                res_pokemon = session.get(f"{url_pokemon}/{res_pokemon_sp["id"]}/",timeout=3).json()
            except requests.exceptions.RequestException as e:
                print(f"Error de red del endpoint 'pokemon' en la especie {species['name']}, saltando... \nDetalle: {e}")
                continue #Si falla que salte al siguiente elemento del bucle
            else:
                #2.3. Extraemos las variables de interés y guardamos en el dict

                ##Extraemos variables del endpoint pokemon-species y guardamos en dict
                datos_pokeapi["id"].append(res_pokemon_sp["id"])
                datos_pokeapi["name"].append(res_pokemon_sp["name"])
                datos_pokeapi["generation"].append(res_pokemon_sp["generation"]["name"]) #generación (categorica)
                datos_pokeapi["shape"].append(res_pokemon_sp["shape"]["name"]) #forma (categorica)
                datos_pokeapi["capture_rate"].append(res_pokemon_sp["capture_rate"]) #rate captura (integer); hasta 255, cuanto mayor más fácil de cazar

                ##Extraemos variables directas del endpoint pokemon y guardamos en dict
                datos_pokeapi["height"].append(res_pokemon["height"]) #height in decimetres
                datos_pokeapi["weight"].append(res_pokemon["weight"]) #weight in hectograms
                ## Procesamos stats (siempre tendremos los 6 stats) y guardamos stats en dict
                pokemon_stats = {f"stat_{s['stat']['name']}": s["base_stat"] for s in res_pokemon["stats"]}
                datos_pokeapi["stat_hp"].append(pokemon_stats["stat_hp"])
                datos_pokeapi["stat_attack"].append(pokemon_stats["stat_attack"])
                datos_pokeapi["stat_special-attack"].append(pokemon_stats["stat_special-attack"])
                datos_pokeapi["stat_defense"].append(pokemon_stats["stat_defense"])
                datos_pokeapi["stat_special-defense"].append(pokemon_stats["stat_special-defense"])
                datos_pokeapi["stat_speed"].append(pokemon_stats["stat_speed"])
                ## Procesamos types y guardamos types en dict
                datos_pokeapi["type_1"].append(res_pokemon["types"][0]["type"]["name"]) #Siempre habrá un tipo primario
                datos_pokeapi["type_2"].append(res_pokemon["types"][1]["type"]["name"] if len(res_pokemon["types"]) > 1 else None) #tipo secundario, puede ser NULL

    # Pausa de cortesía de 50 milisegundos para respetar el servidor
    time.sleep(0.05)
#Mensaje final
print("3. Extracción completada!")

1. Solicitando lista de especies a PokeAPI...
2. Procesando y filtrando Pokémon de Nivel 1 (Formas Base)...
3. Extracción completada!


In [ ]:
#Comprobaciones iniciales de datos
# 1. Creamos el DataFrame a partir del diccionario recolectado
df_pokemon = pd.DataFrame(datos_pokeapi)

# 2. Comprobamos dimensiones y tipos de datos
print(f"Dimensiones del DataFrame: {df_pokemon.shape}")
print("\nTipos de datos por columna:")
print(df_pokemon.dtypes)

# 3. Veamos info general
print("")
print(df_pokemon.info())

# 4. Mostramos los 10 primeros elementos del DataFrame
df_pokemon.head(10)

Dimensiones del DataFrame: (541, 15)

Tipos de datos por columna:
id                      int64
name                      str
generation                str
shape                     str
capture_rate            int64
height                  int64
weight                  int64
stat_hp                 int64
stat_attack             int64
stat_special-attack     int64
stat_defense            int64
stat_special-defense    int64
stat_speed              int64
type_1                    str
type_2                    str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id                    541 non-null    int64
 1   name                  541 non-null    str  
 2   generation            541 non-null    str  
 3   shape                 541 non-null    str  
 4   capture_rate          541 non-null    int64
 5   height                541 n

,id,name,generation,shape,capture_rate,height,weight,stat_hp,stat_attack,stat_special-attack,stat_defense,stat_special-defense,stat_speed,type_1,type_2
0,1,bulbasaur,generation-i,quadruped,45,7,69,45,49,65,49,65,45,grass,poison
1,4,charmander,generation-i,upright,45,6,85,39,52,60,43,50,65,fire,NaN
2,7,squirtle,generation-i,upright,45,5,90,44,48,50,65,64,43,water,NaN
3,10,caterpie,generation-i,armor,255,3,29,45,30,20,35,20,45,bug,NaN
4,13,weedle,generation-i,armor,255,3,32,40,35,20,30,20,50,bug,poison
5,16,pidgey,generation-i,wings,255,3,18,40,45,35,40,35,56,normal,flying
6,19,rattata,generation-i,quadruped,255,3,35,30,56,25,35,35,72,normal,NaN
7,21,spearow,generation-i,wings,255,3,20,40,60,31,30,31,70,normal,flying
8,23,ekans,generation-i,squiggle,255,20,69,35,60,40,44,54,55,poison,NaN
9,27,sandshrew,generation-i,upright,255,6,120,50,75,20,85,30,40,ground,NaN


In [9]:
#Guardamos el resultado en CSV
df_pokemon.to_csv("./data/df_pokeapi_raw.csv", sep=",")